## (01) Install & Import Dependencies

In [1]:
import os, re, time, random, warnings
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from collections import defaultdict

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

import nltk
from nltk.tokenize import word_tokenize
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

import gensim
from gensim.models import Word2Vec, FastText, KeyedVectors
import gensim.downloader as gensim_api

from transformers import BertTokenizerFast, BertModel, AutoTokenizer
from torchcrf import CRF
from seqeval.metrics import f1_score, classification_report

warnings.filterwarnings('ignore')

# ─── Reproducibility ─────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
elif torch.backends.mps.is_available():
    DEVICE = torch.device('mps')   # ← Apple GPU acceleration
else:
    DEVICE = torch.device('cpu')

print(f'Using device: {DEVICE}')

Using device: mps


## (02) Load BC5CDR Dataset

In [6]:
# ── Set your paths here ──────────────────────────────────────────────────────
TRAIN_FILE = 'train.txt'
VAL_FILE   = 'val.txt'
TEST_FILE  = 'test.txt'

# ─── Parser ───────────────────────────────────────────────────────────────────
def read_conll(filepath):
    """Read CoNLL-style file. Returns list of (tokens, tags) tuples.
    Handles both TAB and SPACE separators, and skips doc-start lines."""
    sentences, tokens, tags = [], [], []
    with open(filepath, encoding='utf-8') as f:
        for line in f:
            line = line.rstrip('\n')
            if line.startswith('-DOCSTART-') or line.startswith('#'):
                continue
            if line.strip() == '':
                if tokens:
                    sentences.append((tokens, tags))
                    tokens, tags = [], []
            else:
                parts = line.split('\t') if '\t' in line else line.split()
                if len(parts) >= 2:
                    tokens.append(parts[0])
                    tags.append(parts[-1])   # last column = NER tag
    if tokens:
        sentences.append((tokens, tags))
    return sentences

train_data = read_conll(TRAIN_FILE)
val_data   = read_conll(VAL_FILE)
test_data  = read_conll(TEST_FILE)

print(f'Train sentences : {len(train_data)}')
print(f'Val   sentences : {len(val_data)}')
print(f'Test  sentences : {len(test_data)}')
print('\nSample sentence:')
print(train_data[0])

Train sentences : 4560
Val   sentences : 4581
Test  sentences : 4797

Sample sentence:
(['Selegiline', '-', 'induced', 'postural', 'hypotension', 'in', 'Parkinson', "'", 's', 'disease', ':', 'a', 'longitudinal', 'study', 'on', 'the', 'effects', 'of', 'drug', 'withdrawal', '.'], ['O', 'O', 'O', 'B-Disease', 'I-Disease', 'O', 'B-Disease', 'I-Disease', 'I-Disease', 'I-Disease', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O'])



## (03) Build Vocabulary & Tag Mappings

In [7]:
PAD_TOKEN = '<PAD>'
UNK_TOKEN = '<UNK>'

# ─── Build vocab from TRAIN only (standard NLP practice) ─────────────────────
all_train_tokens = [tok for sent, _ in train_data for tok in sent]
vocab = sorted(set(all_train_tokens))
word2idx = {PAD_TOKEN: 0, UNK_TOKEN: 1}
for w in vocab:
    word2idx[w] = len(word2idx)
idx2word = {v: k for k, v in word2idx.items()}

# ─── Tag mappings ─────────────────────────────────────────────────────────────
all_tags = sorted(set(tag for _, tags in train_data for tag in tags))
tag2idx  = {PAD_TOKEN: 0}
for t in all_tags:
    tag2idx[t] = len(tag2idx)
idx2tag = {v: k for k, v in tag2idx.items()}

VOCAB_SIZE  = len(word2idx)
NUM_CLASSES = len(tag2idx)
print(f'Vocabulary size : {VOCAB_SIZE}')
print(f'Tag set         : {list(tag2idx.keys())}')

Vocabulary size : 9983
Tag set         : ['<PAD>', 'B-Disease', 'I-Disease', 'O']


## (4) Tokenizers

In [8]:
from transformers import AutoTokenizer as HFTokenizer

# ─── Load HuggingFace tokenizer once ─────────────────────────────────────────
HF_MODEL_NAME = 'bert-base-cased'
hf_tokenizer  = HFTokenizer.from_pretrained(HF_MODEL_NAME)

# ─────────────────────────────────────────────────────────────────────────────
def whitespace_tokenize(text):
    """Simple whitespace split."""
    return text.split()

def nltk_tokenize(text):
    """NLTK word tokenizer."""
    return word_tokenize(text)

def hf_tokenize(text):
    """HuggingFace BPE/WordPiece tokenizer — returns subword tokens."""
    return hf_tokenizer.tokenize(text)

TOKENIZERS = {
    'Whitespace' : whitespace_tokenize,
    'NLTK'       : nltk_tokenize,
    'BPE_WP'     : hf_tokenize,
}

# ─── Demo ─────────────────────────────────────────────────────────────────────
sample = 'patients with diabetes mellitus were treated'
for name, fn in TOKENIZERS.items():
    print(f'{name:12s}: {fn(sample)}')

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]

Whitespace  : ['patients', 'with', 'diabetes', 'mellitus', 'were', 'treated']
NLTK        : ['patients', 'with', 'diabetes', 'mellitus', 'were', 'treated']
BPE_WP      : ['patients', 'with', 'diabetes', 'me', '##lli', '##tus', 'were', 'treated']


## (5) Dataset & Dataloader 

In [9]:
MAX_LEN = 128

class NERDataset(Dataset):
    """
    Dataset for non-BERT embeddings (Word2Vec, GloVe, FastText).
    Tokenizer is applied per-word to get final sub-word indices or
    word indices, then tags are aligned to first sub-token (BIO kept).
    """
    def __init__(self, data, word2idx, tag2idx, tokenizer_fn, max_len=MAX_LEN):
        self.samples = []
        for tokens, tags in data:
            ids, tag_ids = [], []
            for tok, tag in zip(tokens, tags):
                sub = tokenizer_fn(tok)
                if not sub:
                    sub = [tok]
                for i, s in enumerate(sub):
                    ids.append(word2idx.get(s, word2idx.get(tok, word2idx[UNK_TOKEN])))
                    # Only first sub-token keeps real tag; rest get 'O'
                    tag_ids.append(tag2idx.get(tag if i == 0 else 'O', tag2idx['O']))
            ids    = ids[:max_len]
            tag_ids = tag_ids[:max_len]
            self.samples.append((torch.tensor(ids, dtype=torch.long),
                                  torch.tensor(tag_ids, dtype=torch.long)))

    def __len__(self):  return len(self.samples)
    def __getitem__(self, i): return self.samples[i]


def collate_fn(batch):
    """Pad sequences in a batch to equal length."""
    seqs, tags = zip(*batch)
    seqs_padded = pad_sequence(seqs, batch_first=True, padding_value=0)
    tags_padded = pad_sequence(tags, batch_first=True, padding_value=0)
    mask = (seqs_padded != 0).bool()
    return seqs_padded, tags_padded, mask


def make_loaders(train_data, val_data, test_data, word2idx, tag2idx,
                 tokenizer_fn, batch_size=32):
    tr = NERDataset(train_data, word2idx, tag2idx, tokenizer_fn)
    vl = NERDataset(val_data,   word2idx, tag2idx, tokenizer_fn)
    te = NERDataset(test_data,  word2idx, tag2idx, tokenizer_fn)
    return (
        DataLoader(tr, batch_size=batch_size, shuffle=True,  collate_fn=collate_fn),
        DataLoader(vl, batch_size=batch_size, shuffle=False, collate_fn=collate_fn),
        DataLoader(te, batch_size=batch_size, shuffle=False, collate_fn=collate_fn),
    )

print('Dataset utilities defined.')

Dataset utilities defined.


In [3]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

# Must be before ANY other TF import
import tensorflow as tf
tf.compat.v1.enable_eager_execution()

import tensorflow_hub as hub

print(f'TF version: {tf.__version__}')
print(f'Eager mode: {tf.executing_eagerly()}')  # must print True

print('Loading ELMo...')
elmo_tf = hub.load('https://tfhub.dev/google/elmo/3')
ELMO_DIM = 1024
print(f'ELMo loaded.')

TF version: 2.21.0
Eager mode: True
Loading ELMo...
ELMo loaded.


In [4]:
# Quick test — should print a tensor of shape (1, 5, 1024)
test_tokens = ['patients', 'with', 'diabetes', 'mellitus', 'treated']
result = elmo_tf.signatures['tokens'](
    tokens=tf.constant([test_tokens]),
    sequence_len=tf.constant([len(test_tokens)])
)
emb = result['elmo'].numpy()
print(f'Shape: {emb.shape}')   # should be (1, 5, 1024)
print('ELMo is working correctly!')

Shape: (1, 5, 1024)
ELMo is working correctly!
